# CPTR 480 — Week 1 Wednesday: Board Bring-Up Methodology
### April 1, 2026

**Today's agenda**
1. Lab 1 debrief — what worked, what didn't
2. The bring-up pyramid — a systematic methodology
3. Schematic reading workshop — live walkthrough of the 2026 board
4. Preview of Lab 2

---
## Part 1 — Lab 1 Debrief (10 min)

**Quick check-in:**
- Who finished all 5 parts?
- Where did people get stuck?

**Common issues from yesterday:**
- USB cable was charge-only → Pico never appeared as mass storage
- Forgot to hold BOOTSEL before plugging in
- Cold solder joints → intermittent signal
- `i2c.scan()` returned `[]` — what does that tell you?

> **Discussion:** *"What does an empty I2C scan result mean, and how would you systematically debug it?"*

---
*[Discuss 2–3 minutes, then move on]*

---
## Part 2 — The Bring-Up Pyramid (20 min)

Every piece of hardware you ever touch follows the same debug order. Skipping layers wastes hours.

```
        ┌─────────────┐
        │ Application │   ← Only once everything below works
        ├─────────────┤
        │Communication│   ← I2C scan, SPI loopback
        ├─────────────┤
        │   Clocks    │   ← ICs won't respond without supply clock
        ├─────────────┤
        │ Power Rails │   ← Wrong voltage can destroy hardware
        ├─────────────┤
        │Visual Insp. │   ← Bridges, orientation, missing parts
        └─────────────┘
```

### Layer 1 — Visual Inspection
Before **any** power is applied:
- Check component orientation (caps, ICs with pin 1 markers)
- Look for solder bridges between adjacent pads
- Verify no missing components (especially decoupling caps you may not have placed)
- Use a loupe or phone camera — your eyes will miss 0.5mm bridges

### Layer 2 — Power Rails
**"Measure twice, power-on once."**

| Rail | Expected | Why it matters |
|------|----------|----------------|
| VBUS | 5.0 V | USB power in |
| 3V3  | 3.3 V | Logic supply for Pico, OLED, Si5351a |
| AVDD | 3.3 V (analog) | PCM1808 analog supply — noise-sensitive |

- Measure **before** connecting any expensive peripheral
- A 5V rail on a 3.3V device will destroy it instantly

### Layer 3 — Clocks
Many ICs **will not respond on I2C or SPI** unless their clock is running.

- The Si5351A has an internal oscillator but needs proper I2C configuration before it outputs clocks
- The PCM1808 needs MCLK before `i2c.scan()` will return its address
- Rule: if you try to communicate and get no response, verify the IC's power and clock source before blaming your code

### Layer 4 — Communication
I2C scan is **diagnostic gold**:
- A device at the right address = power ✓, clock ✓, wiring ✓
- No response = one of those three has failed (or wrong address)
- Wrong address = firmware bug or address pin wired wrong

```python
# MicroPython quick scan — run this first
import machine
i2c = machine.I2C(0, sda=machine.Pin(4), scl=machine.Pin(5))
print("Devices found:", [hex(a) for a in i2c.scan()])
# Si5351A should be at 0x60, OLED at 0x3C
```

### Layer 5 — Application
Only start your full application when layers 1–4 check out.

> **Discussion:** *"I once spent two full days debugging SPI communication that failed because the MISO line had a 10kΩ pull-up to the wrong voltage rail. At which layer would systematic bring-up have caught that in 5 minutes?"*

---
## Part 3 — Schematic Reading Workshop (15 min)

*[Open the 2026 board schematic in KiCad or PDF viewer on the projector]*

### How the 2026 Board Schematic is Organized

The schematic is **hierarchical** — one top-level sheet links to sub-sheets:
- `raspberry_pi_pico.kicad_sch` — the YD-RP2040 and its GPIO assignments
- `si5351a.kicad_sch` — local oscillator for the QSD mixer
- `pcm1808.kicad_sch` — audio ADC (I2S output to Pico)
- `qsd-sdr.kicad_sch` — quadrature sampling detector (the mixer itself)
- `oled.kicad_sch` — SSD1306 display

### How to Follow a Net Across Sheets

Net labels are global. Find `I2C_SDA` on the Pico sheet — then use **Edit → Find** (or `Ctrl+F` in KiCad) to jump to where the same net appears on the Si5351a sheet and OLED sheet.

**Exercise:** Find `MCLK` on the schematic. Which pin on the Pico drives it? What is the frequency it needs to output? *(Answer: GPIO 21 via PIO, 24.576 MHz for the 2026 board)*

### Reading a Voltage Divider

The PCM1808 analog input needs a **bias point** — the AC audio signal rides on a DC offset.

$$V_{bias} = V_{DD} \times \frac{R_2}{R_1 + R_2}$$

For our board: $V_{DD} = 3.3\ \text{V}$, and the divider is designed to set $V_{bias} \approx 1.65\ \text{V}$ (half-supply).

| Symbol | What it tells you |
|--------|-------------------|
| `AGND` | Analog ground — separate from digital GND to reduce noise coupling |
| `GND`  | Digital/power ground |
| `AVDD` | Analog supply — filtered separately from the digital 3.3V rail |

> **Key takeaway:** Analog and digital grounds are connected at **one point only** (a "star ground"). Routing high-current digital return currents through the analog ground plane injects noise into the ADC — this is a classic PCB layout mistake.

### What to Look for When Debugging a New Board

1. Check that decoupling caps (100nF ceramic) are placed right next to each IC's VDD pin
2. Verify pull-up resistors on I2C lines (typically 4.7kΩ to 3.3V)
3. Confirm GPIO numbers in firmware match the schematic net names

In [ ]:
# Live demo: voltage divider calculator
# Students can run this and experiment with resistor values

R1 = 100e3   # ohms
R2 = 100e3   # ohms
VDD = 3.3    # volts

V_bias = VDD * R2 / (R1 + R2)
print(f"R1 = {R1/1e3:.1f} kΩ,  R2 = {R2/1e3:.1f} kΩ")
print(f"V_bias = {V_bias:.3f} V  (target: {VDD/2:.3f} V)")
print(f"Headroom above bias: {VDD - V_bias:.3f} V")
print(f"Max signal swing (±): {min(V_bias, VDD - V_bias) * 1000:.0f} mV peak")

---
## Part 4 — Preview: Lab 2 (5 min)

**Lab 2 (Tuesday, April 7):** Pico Debug Stack Setup + Full Board Validation + README

You need **two things ready before you walk in:**

### 1. Your HW 1 README skeleton
Lab 2 asks you to fill in a pin assignment table for every peripheral on the board. If you show up without a starting skeleton, you'll spend the first 45 minutes writing Markdown tables instead of doing the lab.

**This week:** Open the schematic, find each peripheral's GPIO connections, and fill in the table now.

| Peripheral | GPIO | Direction | Protocol |
|------------|------|-----------|----------|
| OLED SDA   | ?    | Out       | I2C      |
| OLED SCL   | ?    | Out       | I2C      |
| Si5351a SDA| ?    | Out       | I2C      |
| PCM1808 MCLK| ?   | Out       | PIO/Clock|
| ...        | ...  | ...       | ...      |

### 2. Your toolchain must work
Lab 2 validates the **C/C++ Pico SDK** toolchain. If `cmake` can't configure or `arm-none-eabi-gcc` is not found, you'll be stuck at layer 0 while everyone else is working.

**Before Tuesday:** Build the `blink` example from the Pico SDK. If it compiles and flashes, you're ready.

---

## Summary

| Layer | Tool | Pass condition |
|-------|------|----------------|
| Visual | Eyes + loupe | No bridges, correct orientation |
| Power | Multimeter | 3.3V on 3V3, 5V on VBUS |
| Clocks | Scope / logic analyzer | MCLK toggling at 24.576 MHz |
| Communication | `i2c.scan()` | 0x3C (OLED), 0x60 (Si5351a) visible |
| Application | Your code | Correct behavior |

**HW 1 is due Monday April 6 at 11:00 a.m.** — schematic questions + README skeleton.